# 3D U-Net v2 Training — R Only
**Step 1:** Mount Drive (Python cell — just for file access)
**Step 2:** Install R + torch
**Step 3:** Train in R

Runtime: **Python** (but training runs in R via subprocess)

In [ ]:
# CELL 1: Mount Drive and copy data locally (fast)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/R_project/data/training'
dst = '/content/training'
if not os.path.exists(dst):
    print('Copying training data to local disk (faster than reading from Drive)...')
    shutil.copytree(src, dst)
print('Files:', os.listdir(dst))
print('MRI count:', len(os.listdir(f'{dst}/mri')))
print('SEG count:', len(os.listdir(f'{dst}/seg')))
print('Done! Data copied to /content/training/')

In [ ]:
# CELL 2: Install R and R torch
%%bash
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq r-base r-base-dev > /dev/null 2>&1
echo 'R installed'
R --version | head -1

In [ ]:
# CELL 3: Install R packages
%%bash
R -e "
options(timeout = 600)
install.packages(c('torch', 'coro', 'readr'), repos = 'https://cloud.r-project.org', quiet = TRUE)
library(torch)
install_torch(type = 'cpu', reinstall = TRUE)
cat('Torch installed:', torch_is_installed(), '\n')
cat('Test:', as.numeric(torch_tensor(1) + torch_tensor(1)), '\n')
"

In [ ]:
# CELL 4: Write R training script
%%writefile /content/train.R

library(torch)
library(coro)
library(readr)

TRAINING_DIR <- '/content/training'
MODEL_DIR <- '/content/drive/MyDrive/R_project/models'
dir.create(MODEL_DIR, showWarnings = FALSE, recursive = TRUE)

# ── 3D U-Net ──

conv_block_3d <- nn_module('ConvBlock3D',
  initialize = function(in_ch, out_ch) {
    self$conv1 <- nn_conv3d(in_ch, out_ch, 3, padding = 1)
    self$bn1 <- nn_batch_norm3d(out_ch)
    self$conv2 <- nn_conv3d(out_ch, out_ch, 3, padding = 1)
    self$bn2 <- nn_batch_norm3d(out_ch)
    self$relu <- nn_relu()
  },
  forward = function(x) {
    x <- self$relu(self$bn1(self$conv1(x)))
    self$relu(self$bn2(self$conv2(x)))
  }
)

encoder_block <- nn_module('Encoder',
  initialize = function(in_ch, out_ch) {
    self$conv <- conv_block_3d(in_ch, out_ch)
    self$pool <- nn_max_pool3d(2)
  },
  forward = function(x) {
    feat <- self$conv(x)
    list(f = feat, p = self$pool(feat))
  }
)

decoder_block <- nn_module('Decoder',
  initialize = function(in_ch, out_ch) {
    self$up <- nn_conv_transpose3d(in_ch, out_ch, 2, stride = 2)
    self$conv <- conv_block_3d(out_ch * 2L, out_ch)
  },
  forward = function(x, skip) {
    x <- self$up(x)
    dd <- skip$size(3) - x$size(3)
    dh <- skip$size(4) - x$size(4)
    dw <- skip$size(5) - x$size(5)
    x <- nnf_pad(x, c(dw %/% 2, dw - dw %/% 2, dh %/% 2, dh - dh %/% 2, dd %/% 2, dd - dd %/% 2))
    self$conv(torch_cat(list(x, skip), dim = 2))
  }
)

unet_3d <- nn_module('UNet3D',
  initialize = function(in_ch = 4L, n_cls = 4L, base = 32L) {
    self$e1 <- encoder_block(in_ch, base)
    self$e2 <- encoder_block(base, base*2L)
    self$e3 <- encoder_block(base*2L, base*4L)
    self$e4 <- encoder_block(base*4L, base*8L)
    self$bn <- conv_block_3d(base*8L, base*16L)
    self$d4 <- decoder_block(base*16L, base*8L)
    self$d3 <- decoder_block(base*8L, base*4L)
    self$d2 <- decoder_block(base*4L, base*2L)
    self$d1 <- decoder_block(base*2L, base)
    self$final <- nn_conv3d(base, n_cls, 1)
  },
  forward = function(x) {
    e1 <- self$e1(x); e2 <- self$e2(e1$p)
    e3 <- self$e3(e2$p); e4 <- self$e4(e3$p)
    b <- self$bn(e4$p)
    self$final(self$d1(self$d2(self$d3(self$d4(b, e4$f), e3$f), e2$f), e1$f))
  }
)

# ── Dataset ──

augment_3d <- function(mri, seg) {
  if (runif(1) > 0.5) { mri <- torch_flip(mri, 2); seg <- torch_flip(seg, 1) }
  if (runif(1) > 0.5) { mri <- torch_flip(mri, 3); seg <- torch_flip(seg, 2) }
  if (runif(1) > 0.5) { mri <- torch_flip(mri, 4); seg <- torch_flip(seg, 3) }
  if (runif(1) > 0.5) mri <- mri * (0.9 + runif(1) * 0.2) + (runif(1) - 0.5) * 0.2
  if (runif(1) > 0.7) mri <- mri + torch_randn_like(mri) * 0.05
  list(mri = mri, seg = seg)
}

tumor_ds <- dataset('TumorDS',
  initialize = function(paths_mri, paths_seg, aug = FALSE) {
    self$m <- paths_mri; self$s <- paths_seg; self$aug <- aug
  },
  .getitem = function(i) {
    mri <- torch_load(self$m[i])$to(dtype = torch_float())
    seg <- torch_load(self$s[i])$to(dtype = torch_long())
    if (mri$dim() == 3) mri <- mri$unsqueeze(1)
    if (mri$dim() == 4 && mri$size(4) <= 4) mri <- mri$permute(c(4,1,2,3))
    if (self$aug) { a <- augment_3d(mri, seg); mri <- a$mri; seg <- a$seg }
    list(x = mri, y = seg)
  },
  .length = function() length(self$m)
)

# ── Loss ──

CW <- c(0.5, 3.0, 1.5, 2.0)

w_dice <- function(pred, target) {
  ps <- nnf_softmax(pred, dim = 2)
  total <- torch_tensor(0, device = pred$device)
  for (c in 1:4) {
    pc <- ps[, c,,,]; tc <- (target == (c-1L))$to(dtype = torch_float())
    total <- total + CW[c] * (1 - (2*(pc*tc)$sum()+1) / (pc$sum()+tc$sum()+1))
  }
  total / sum(CW)
}

loss_fn <- function(pred, target) {
  w <- torch_tensor(CW, dtype = torch_float(), device = pred$device)
  0.4 * nnf_cross_entropy(pred, target$to(dtype=torch_long()), weight=w) + 0.6 * w_dice(pred, target)
}

dice_scores <- function(pred, target) {
  pc <- pred$argmax(dim = 2)$squeeze(1)
  d <- numeric(4); names(d) <- c('bg','nec','edema','enh')
  for (c in 1:4) {
    p <- (pc == (c-1L)); t <- (target == (c-1L))
    inter <- (p & t)$sum()$item(); tot <- p$sum()$item() + t$sum()$item()
    d[c] <- if (tot > 0) 2*inter/tot else NA_real_
  }
  d
}

# ── Load data ──

manifest <- read_csv(file.path(TRAINING_DIR, 'manifest.csv'), show_col_types = FALSE)
manifest <- manifest[manifest$tumor_voxels > 0, ]
cat('Samples:', nrow(manifest), '\n')

manifest$mf <- file.path(TRAINING_DIR, manifest$mri_file)
manifest$sf <- file.path(TRAINING_DIR, manifest$seg_file)

set.seed(42)
pts <- unique(manifest$patient_id)
tp <- sample(pts, floor(length(pts)*0.8))
vp <- setdiff(pts, tp)
tr <- manifest[manifest$patient_id %in% tp, ]
va <- manifest[manifest$patient_id %in% vp, ]
cat('Train:', nrow(tr), '| Val:', nrow(va), '\n')

train_dl <- dataloader(tumor_ds(tr$mf, tr$sf, TRUE), batch_size=2L, shuffle=TRUE)
val_dl <- dataloader(tumor_ds(va$mf, va$sf, FALSE), batch_size=2L)

# ── Train ──

model <- unet_3d(4L, 4L, 32L)
np <- sum(sapply(model$parameters, function(p) prod(p$shape)))
cat('Params:', format(np, big.mark=','), '\n')

opt <- optim_adam(model$parameters, lr = 3e-4, weight_decay = 1e-5)
cosine <- function(ep, mx) 1e-6 + 0.5*(3e-4-1e-6)*(1+cos(pi*ep/mx))

best <- 0; pat <- 0; EPOCHS <- 150L; PAT <- 20L
hist <- data.frame()

# Resume
hp <- file.path(MODEL_DIR, 'unet_v2_history.csv')
cp <- file.path(MODEL_DIR, 'unet_v2_checkpoint.pt')
if (file.exists(hp)) {
  hist <- read_csv(hp, show_col_types=FALSE)
  best <- max(hist$val_dice, na.rm=TRUE)
  cat('Resume from epoch', nrow(hist)+1, '| Best:', round(best,4), '\n')
}
if (file.exists(cp)) tryCatch({model$load_state_dict(torch_load(cp)); cat('Checkpoint loaded\n')}, error=function(e) NULL)

se <- nrow(hist) + 1L
t0 <- Sys.time()

cat(sprintf('%-5s | %-8s | %-8s | %-7s | %-7s | %-7s | %-7s\n', 'Ep','TrLoss','VaLoss','Dice','Nec','Edema','Enh'))
cat(strrep('=', 65), '\n')

for (epoch in seq(se, EPOCHS)) {
  lr <- cosine(epoch, EPOCHS)
  for (pg in opt$param_groups) pg$lr <- lr

  model$train(); tl <- c()
  coro::loop(for (b in train_dl) {
    opt$zero_grad()
    loss <- loss_fn(model(b$x), b$y)
    loss$backward()
    nn_utils_clip_grad_norm_(model$parameters, 1.0)
    opt$step()
    tl <- c(tl, loss$item())
  })

  model$eval(); vl <- c(); ad <- list()
  with_no_grad({
    coro::loop(for (b in val_dl) {
      pred <- model(b$x)
      vl <- c(vl, loss_fn(pred, b$y)$item())
      ad <- c(ad, list(dice_scores(pred, b$y)))
    })
  })

  dm <- do.call(rbind, ad)
  dn <- mean(dm[,'nec'], na.rm=T)
  de <- mean(dm[,'edema'], na.rm=T)
  dh <- mean(dm[,'enh'], na.rm=T)
  dd <- mean(c(dn,de,dh), na.rm=T)

  mk <- ''
  if (dd > best) {
    best <- dd
    torch_save(model$state_dict(), file.path(MODEL_DIR, 'unet_3d_v2_best.pt'))
    pat <- 0; mk <- ' *BEST'
  } else pat <- pat + 1

  torch_save(model$state_dict(), cp)
  hist <- rbind(hist, data.frame(epoch=epoch, train_loss=mean(tl), val_loss=mean(vl),
    val_dice=dd, dice_nec=dn, dice_edema=de, dice_enh=dh, lr=lr))
  write_csv(hist, hp)

  el <- as.numeric(difftime(Sys.time(), t0, units='mins'))
  cat(sprintf('%-5d | %-8.4f | %-8.4f | %-7.4f | %-7.4f | %-7.4f | %-7.4f [%.0fm]%s\n',
    epoch, mean(tl), mean(vl), dd, dn, de, dh, el, mk))

  if (pat >= PAT) { cat('Early stopping\n'); break }
  if (epoch %% 10 == 0) gc()
}

# ── Save JIT ──

cat('\nSaving JIT model...\n')
model$load_state_dict(torch_load(file.path(MODEL_DIR, 'unet_3d_v2_best.pt')))
model$eval()
dummy <- torch_randn(1, 4, 48, 48, 48)
with_no_grad({ traced <- jit_trace(model, dummy) })
jit_save(traced, file.path(MODEL_DIR, 'unet_3d_v2_jit.pt'))

cat('\nDONE! Best Dice:', round(best, 4), '\n')
cat('Model at:', file.path(MODEL_DIR, 'unet_3d_v2_jit.pt'), '\n')


In [ ]:
# CELL 5: Run R training
%%bash
cd /content && R --no-save < train.R 2>&1 | tee training_log.txt

In [ ]:
# CELL 6: Check results
import os
model_dir = '/content/drive/MyDrive/R_project/models'
for f in os.listdir(model_dir):
    size = os.path.getsize(os.path.join(model_dir, f)) / 1e6
    print(f'{f}: {size:.1f} MB')